# ProtT5 Embedding + Machine Learning (Kaggle)

Notebook ini menjalankan pipeline `EmbeddingML/`: peptida diubah menjadi embedding menggunakan **ProtT5 beku**, lalu digunakan untuk melatih dan mengevaluasi **SVM** atau **Random Forest**. Implementasi preprocessing, split, model, dan evaluasi tetap memakai modul proyek (bukan salinan kode di notebook).

**Alur:** setup repository/data → install dependensi → pilih eksperimen → validasi konfigurasi → hitung/cache embedding → split → training dan evaluasi → tampilkan metrik, plot, dan artefak.

> Aktifkan **Internet** pada pengaturan Kaggle untuk mengunduh model ProtT5 dan dependensi. GPU sangat disarankan; inference ProtT5-XL memerlukan waktu dan memori yang besar.

## 1. Setup repository dan data

Jika notebook dijalankan dari repository yang sudah tersedia, path tersebut dipakai. Jika belum, notebook akan meng-clone repository ke `/kaggle/working/B-Cell-Epitopes`. Pastikan CSV dataset tersedia di `dataset/` atau ubah `DATA_DIR` pada sel konfigurasi.

In [ ]:
from pathlib import Path
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "EmbeddingML" / "train.py").is_file():
    PROJECT_ROOT = Path("/kaggle/working/B-Cell-Epitopes")
    if not (PROJECT_ROOT / "EmbeddingML" / "train.py").is_file():
        PROJECT_ROOT.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(
            [
                "git", "clone",
                "https://github.com/azhar416/B-Cell-Epitopes.git",
                str(PROJECT_ROOT),
            ],
            check=True,
        )

if not (PROJECT_ROOT / "EmbeddingML" / "train.py").is_file():
    raise FileNotFoundError(
        f"EmbeddingML/train.py tidak ditemukan di {PROJECT_ROOT}. "
        "Pastikan repository sudah ter-clone dengan benar."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "dataset"
print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset directory: {DATA_DIR}")
print(f"Dataset directory exists: {DATA_DIR.is_dir()}")

## 2. Install dependensi

Jalankan sekali per sesi Kaggle. Sel ini menggunakan dependency manifest proyek.

In [ ]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(PROJECT_ROOT / "EmbeddingML" / "requirements.txt"),
    ],
    check=True,
)

## 3. Konfigurasi eksperimen

Ubah `DATASET_NAME` menjadi `"sars"` untuk dataset SARS dan `MODEL_TYPE` menjadi `"random_forest"` untuk Random Forest. `USE_BIOLOGICAL_FEATURES=True` menggabungkan delapan descriptor biologis dengan embedding ProtT5. Untuk dataset Kaggle yang di-attach terpisah, arahkan `DATA_DIR` ke folder berisi `input_bcell.csv` dan/atau `input_sars.csv`.

In [ ]:
from EmbeddingML.CFG import CFG

DATASET_NAME = "bcell"  # "bcell" atau "sars"
MODEL_TYPE = "svm"  # "svm" atau "random_forest"
USE_BIOLOGICAL_FEATURES = False
DEVICE = "auto"  # "auto", "cpu", atau "cuda"
BATCH_SIZE = 4  # turunkan, misalnya ke 1 atau 2, jika GPU kehabisan memori
VALIDATION_SIZE = 0.2
RANDOM_STATE = 42

if not DATA_DIR.is_dir():
    raise FileNotFoundError(
        f"Folder data tidak ditemukan: {DATA_DIR}. "
        "Atur DATA_DIR ke direktori dataset Kaggle yang berisi file CSV."
    )

OUTPUT_DIR = (
    Path("/kaggle/working/EmbeddingML-artifacts")
    if Path("/kaggle/working").is_dir()
    else PROJECT_ROOT / "EmbeddingML" / "artifacts"
)

config = CFG(
    DATASET_NAME=DATASET_NAME,
    DATA_DIR=DATA_DIR,
    OUTPUT_DIR=OUTPUT_DIR,
    VALIDATION_SIZE=VALIDATION_SIZE,
    RANDOM_STATE=RANDOM_STATE,
    DEVICE=DEVICE,
    BATCH_SIZE=BATCH_SIZE,
    USE_BIOLOGICAL_FEATURES=USE_BIOLOGICAL_FEATURES,
    MODEL_TYPE=MODEL_TYPE,
)
config.validate()

expected_csv = DATA_DIR / ("input_bcell.csv" if DATASET_NAME == "bcell" else "input_sars.csv")
if not expected_csv.is_file():
    raise FileNotFoundError(
        f"CSV untuk dataset '{DATASET_NAME}' tidak ditemukan: {expected_csv}. "
        "Pastikan DATA_DIR menunjuk ke folder yang benar."
    )

print(f"Dataset: {config.DATASET_NAME} ({expected_csv})")
print(f"Model: {config.MODEL_TYPE}; biological features: {config.USE_BIOLOGICAL_FEATURES}")
print(f"Device: {config.DEVICE}; batch size: {config.BATCH_SIZE}")
print(f"Output directory: {OUTPUT_DIR}")

## 4. Makna split dan pemeriksaan runtime

- **B-cell:** stratified 5-fold cross-validation pada level protein; protein tidak muncul di train dan validation pada fold yang sama.
- **SARS:** data saat ini hanya memiliki satu protein, sehingga protein-level CV tidak memungkinkan. Pipeline menggunakan stratified peptide-level 5-fold CV; hasil ini **bukan** evaluasi generalisasi pada protein yang belum pernah dilihat.

Encoder ProtT5 dibekukan dan tidak menggunakan LoRA. Jalankan sel berikut untuk memeriksa perangkat yang terdeteksi sebelum inference.

In [ ]:
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
elif config.DEVICE.startswith("cuda"):
    raise RuntimeError(
        "DEVICE meminta CUDA tetapi GPU tidak tersedia. Aktifkan GPU pada "
        "Kaggle Settings atau ubah DEVICE menjadi 'auto' atau 'cpu'."
    )

print("Config valid. Encoder:", config.ENCODER_NAME)
print("Tokenizer:", config.TOKENIZER_NAME)

## 5. Training dan evaluasi

Inference pertama akan mengunduh ProtT5-XL dan menghitung embedding untuk seluruh peptide sequence yang sudah dibersihkan; proses ini bisa lama. Embedding yang sudah dihitung disimpan sebagai cache dan digunakan kembali pada eksperimen berikutnya jika urutan sequence serta nama model/tokenizer cocok. Jangan menghentikan cell sebelum selesai.

In [ ]:
from EmbeddingML.train import run_training

report = run_training(config)
model_output_dir = Path(report["artifact_directory"])
metrics_path = model_output_dir / "cross_validation_metrics.json"
print(f"\nMetrics JSON: {metrics_path}")

## 6. Hasil dan evaluasi visual

In [ ]:
import pandas as pd
from IPython.display import Image, display

summary = {
    "dataset": report["dataset_name"],
    "model": report["model_type"],
    "split_strategy": report["split_strategy"],
    "n_splits": report["n_splits"],
    "validation_size": report["validation_size"],
    "train_validation_proteins_overlap": report["train_validation_proteins_overlap"],
}
display(pd.Series(summary, name="value").to_frame())

print("Split interpretation:", report["validation_metrics_interpretation"])
print("Pooled out-of-fold validation metrics:")
display(pd.Series(report["validation_metrics"], name="value").to_frame())
print("Mean/std validation metrics across folds:")
display(pd.DataFrame({"mean": report["mean_validation_metrics"], "std": report["std_validation_metrics"]}))
print("Mean training metrics across folds (in-sample):")
display(pd.DataFrame({"mean": report["mean_training_metrics"], "std": report["std_training_metrics"]}))
print("Per-fold training and evaluation metrics:")
display(pd.DataFrame([
    {
        "fold": item["fold"],
        **{f"training_{name}": value for name, value in item["training_metrics"].items()},
        **{f"evaluation_{name}": value for name, value in item["validation_metrics"].items()},
    }
    for item in report["fold_metrics"]
]))

for plot_name, plot_path in report["plots"].items():
    print(f"{plot_name}: {plot_path}")
    display(Image(filename=plot_path))

In [ ]:
artifact_paths = sorted(
    path for path in model_output_dir.rglob("*") if path.is_file()
)
print(f"Artifacts ({len(artifact_paths)} files) in {model_output_dir}:")
for path in artifact_paths:
    print(f"- {path}")

print(f"\nEmbedding cache: {OUTPUT_DIR / config.DATASET_NAME / 'prott5_embeddings.npz'}")
print("Simpan output notebook dengan Kaggle 'Save Version' agar artefak di /kaggle/working tersimpan sebagai output run.")

## Catatan penggunaan

- Pastikan **Internet** aktif agar Hugging Face dapat menyediakan ProtT5 pada run pertama. Aktifkan **GPU accelerator** dari pengaturan notebook untuk inference lebih cepat.
- Jika GPU kehabisan memori, ubah `BATCH_SIZE` menjadi `2` atau `1`; inference CPU didukung tetapi dapat sangat lambat.
- Hasil model, prediksi, metrik, grafik, split assignment, dan cache embedding ditulis di `/kaggle/working/EmbeddingML-artifacts/`. Simpan versi notebook dengan output untuk mempertahankan artefak Kaggle.
- Untuk membandingkan SVM dengan Random Forest, ubah `MODEL_TYPE` lalu jalankan ulang. Cache embedding akan dipakai ulang; model dan evaluasi disimpan di subfolder model masing-masing.
- Evaluasi validation bukan test set independen untuk pemilihan model berulang. Catat konfigurasi dan hindari memilih model/threshold berdasarkan validation berulang tanpa menyediakan test set terpisah.